In [15]:
import pandas as pd
z = pd.read_excel("C:/Users/Visiteur.SEGK205-3181VB6/Downloads/Projet_ALM_M2_IFIM(2).xlsx", sheet_name="Loi découlement")


In [25]:
z.columns

Index(['Bilan', 'Catégories Bilan', 'Poste du bilan', 'Montant (en k€)',
       'Taux d'intérèt moyen', 'Loi d'écoulement en liquidité',
       'Durée moyenne (en mois)', 'Loi d'écoulement en taux'],
      dtype='object')

In [5]:
z.columns = range(z.shape[1])
z.columns = z.iloc[0]
z = z.iloc[1:].reset_index(drop=True)
z = z.iloc[:, :-1]
z.columns = z.columns.astype(str).str.strip()

In [6]:
a = range(3, 8)
for i in a:
    print(i)

3
4
5
6
7


In [6]:
z

,Maturité,Zero Coupon
0,1 mois,0.051698
1,3 mois,0.068298
2,6 mois,0.070914
3,9 mois,0.077677
4,1 an,0.072142
5,2 ans,0.070518
6,3 ans,0.076427
7,4 ans,0.076692
8,5 ans,0.076553
9,6 ans,0.075965


In [19]:
import numpy as np
def _pmt(rate_month, nper, pv):
    if nper <= 0:
        return 0.0
    if abs(rate_month) < 1e-12:
        return pv / nper
    return (rate_month * pv) / (1 - (1 + rate_month) ** (-nper))

def _reorder_m_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    base_cols = [c for c in df.columns if not str(c).startswith("M")]
    m_cols = ["M0"] + [f"M{i}" for i in range(1, 121) if f"M{i}" in df.columns]
    ordered = [c for c in base_cols if c in df.columns] + [c for c in m_cols if c in df.columns]
    for c in df.columns:
        if c not in ordered:
            ordered.append(c)
    return df[ordered]


# =========================
# Build empty projection
# =========================
def build_empty_fs_projection(runoff_df: pd.DataFrame) -> pd.DataFrame:
    df = runoff_df.copy()

    if "Loi d'écoulement en taux" in df.columns:
        df = df.drop(columns=["Loi d'écoulement en taux"])

    if "Montant (en k€)" in df.columns:
        df = df.rename(columns={"Montant (en k€)": "M0"})

    if "Taux d'intérèt moyen" in df.columns and "Taux d'intérêt moyen" not in df.columns:
        df = df.rename(columns={"Taux d'intérèt moyen": "Taux d'intérêt moyen"})

    for i in range(1, 121):
        df[f"M{i}"] = np.nan

    return _reorder_m_cols(df)


# =========================

In [ ]:
def Fixed_rate_flows1(financial_statement: pd.DataFrame):
    CRD_flow_df = financial_statement.copy()
    Cash_Flow_df = financial_statement.copy()

    law_col = "Loi d'écoulement en taux"
    dur_col = "Durée moyenne (en mois)"
    rate_col = "Taux d'intérêt moyen"

    CRD_flow_df[dur_col] = pd.to_numeric(CRD_flow_df[dur_col], errors="coerce").fillna(0).astype(int)

    for idx in CRD_flow_df.index:
        M0 = float(pd.to_numeric(CRD_flow_df.at[idx, "M0"], errors="coerce") or 0.0)
        n = int(CRD_flow_df.at[idx, dur_col])
        r = float(CRD_flow_df.at[idx, rate_col]) if rate_col in CRD_flow_df.columns else 0.0
        r_m = r / 12.0
        law = str(CRD_flow_df.at[idx, law_col]).strip().lower() if law_col in CRD_flow_df.columns else ""

        prev_crd = M0

        for i in range(1, 121):
            col = f"M{i}"

            if n <= 0:
                crd_i = 0.0

            elif "linéaire" in law or "lineaire" in law:
                crd_i = M0 * (n - i) / n if i < n else 0.0

            elif "constant" in law:
                if i < n:
                    annuity = _pmt(r_m, n, M0)
                    crd_i = prev_crd * (1 + r_m) - annuity
                    crd_i = max(crd_i, 0.0)
                else:
                    crd_i = 0.0

            elif "in fine" in law or "ine fine" in law:
                crd_i = M0 if i < n else 0.0

            else:
                crd_i = 0.0

            CRD_flow_df.at[idx, col] = crd_i

            cash_i = (-crd_i + prev_crd) if (n > 0 and i < n) else 0.0
            Cash_Flow_df.at[idx, col] = cash_i

            prev_crd = crd_i

    return _reorder_m_cols(CRD_flow_df), _reorder_m_cols(Cash_Flow_df)

In [20]:
k = build_empty_fs_projection(z)
a, b = Fixed_rate_flows1(k)

C:\Users\Visiteur.SEGK205-3181VB6\AppData\Local\Temp\ipykernel_352\243094813.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"M{i}"] = np.nan
C:\Users\Visiteur.SEGK205-3181VB6\AppData\Local\Temp\ipykernel_352\243094813.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"M{i}"] = np.nan
C:\Users\Visiteur.SEGK205-3181VB6\AppData\Local\Temp\ipykernel_352\243094813.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  C

In [23]:
import pandas as pd

# 1) Table BILAN (extrait)
bilan_df = pd.DataFrame({
    "Bilan": ["ACTIF", "ACTIF", "PASSIF"],
    "Categorie": ["CREANCES", "TITRES", "DEPOTS"],
    "Poste du bilan": ["Crédits clientèle", "Obligations", "Dépôts à vue"],
    "Montant (en k€)": [1000, 500, 1200],
})

# 2) Table RUNOFF (extrait) : paramètres d'écoulement
runoff_df = pd.DataFrame({
    "Bilan": ["ACTIF", "PASSIF"],  # ⚠️ il manque "TITRES" ici exprès
    "Categorie": ["CREANCES", "DEPOTS"],
    "Poste du bilan": ["Crédits clientèle", "Dépôts à vue"],
    "Profil": ["constant", "in fine"],
    "Maturité": [36, 12],
})

# 3) MERGE : on recolle les paramètres runoff au bilan
merged_inner = pd.merge(
    bilan_df,
    runoff_df,
    on=["Bilan", "Categorie", "Poste du bilan"],
    how="inner"
)

merged_left = pd.merge(
    bilan_df,
    runoff_df,
    on=["Bilan", "Categorie", "Poste du bilan"],
    how="left"
)

print("=== MERGE INNER (garde seulement les lignes qui matchent) ===")
print(merged_inner)

print("\n=== MERGE LEFT (garde tout le bilan, même sans runoff) ===")
print(merged_left)


=== MERGE INNER (garde seulement les lignes qui matchent) ===
    Bilan Categorie     Poste du bilan  Montant (en k€)    Profil  Maturité
0   ACTIF  CREANCES  Crédits clientèle             1000  constant        36
1  PASSIF    DEPOTS       Dépôts à vue             1200   in fine        12

=== MERGE LEFT (garde tout le bilan, même sans runoff) ===
    Bilan Categorie     Poste du bilan  Montant (en k€)    Profil  Maturité
0   ACTIF  CREANCES  Crédits clientèle             1000  constant      36.0
1   ACTIF    TITRES        Obligations              500       NaN       NaN
2  PASSIF    DEPOTS       Dépôts à vue             1200   in fine      12.0
